# PyTorch パイプライン

このノートブックは PyTorch DLC (Deep Learning Container) を使用したモデルの学習・評価・Pipeline 実行を一気通貫で行います。

### 使用するモデル

PyTorch の `SimpleClassifier` (3 層 MLP: 入力次元 → 64 → 32 → クラス数) を使用します。各隠れ層に Dropout(0.2) を挿入して過学習を抑制します。GPU 推奨ですが、CPU でも動作します。

### DLC 版と BYOC 版の違い

このノートブックは AWS マネージドの PyTorch DLC を使用します。Docker イメージのビルドは不要です。DLC にないライブラリが必要な場合は `pytorch-byoc-pipeline.ipynb` を参照してください。

### このノートブックの流れ

- **Section 1-2**: セットアップとデータ確認
- **Section 3-5**: モデル定義・ローカル学習・ローカル評価
- **Section 6**: MLflow 記録
- **Section 7-8**: SageMaker Training Job / Processing Job
- **Section 9**: SageMaker Pipeline (Train → Register → Evaluate)


### 前提条件

サンプルデータセットは `deploy.sh` の実行時に S3 に自動アップロード済みです。そのまま Notebook を実行できます。

データを変更する場合は `pipelines/container-pytorch-dlc/data/` の CSV ファイルを更新し、以下のコマンドで再アップロードしてください。

```bash
./pipelines/scripts/01-upload-dataset.sh -c container-pytorch-dlc
```


---
## 1. セットアップ

学習・評価に必要なライブラリをインストールします。初回実行時は PyTorch のインストールに数分かかる場合があります。

In [ ]:
# 依存パッケージのインストール
# numpy<2: PyTorch が NumPy 1.x でコンパイルされているため
# torch: CPU 版で再インストール (GPU 版の triton 依存を除去)
!pip install -U "numpy<2" scikit-learn pandas mlflow sagemaker-mlflow matplotlib seaborn
!pip uninstall -y triton triton-nightly 2>/dev/null; pip install --force-reinstall torch --index-url https://download.pytorch.org/whl/cpu


In [ ]:
import os
import io
import json
import logging
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)
logging.getLogger("sagemaker.jumpstart").setLevel(logging.CRITICAL)

プロジェクト設定を定義します。

- `CONTAINER_DIR`: PyTorch DLC コンテナのディレクトリ。SageMaker Job の `source_dir` として使用します
- `DATASET_BUCKET` / `MODEL_BUCKET`: CloudFormation スタックの出力から自動取得します

In [ ]:
# === プロジェクト設定 ===
PROJECT_NAME = "sagemaker-ai-ml-pipeline"
STACK_NAME   = "sagemaker-ai-ml-pipeline-stack"
REGION       = boto3.session.Session().region_name or "us-east-1"
CONTAINER_DIR = "../pipelines/container-pytorch-dlc"

# CloudFormation スタックの出力からバケット名を自動取得
cfn = boto3.client("cloudformation", region_name=REGION)
outputs = {
    o["OutputKey"]: o["OutputValue"]
    for o in cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0]["Outputs"]
}
DATASET_BUCKET = outputs["DatasetBucketName"]
MODEL_BUCKET   = outputs["ModelArtifactBucketName"]

# S3 パス設定 (01-upload-dataset.sh のアップロード先と合わせる)
CONTAINER_TAG = os.path.basename(CONTAINER_DIR)
S3_TRAIN_PREFIX = f"{CONTAINER_TAG}/train"
S3_TEST_PREFIX  = f"{CONTAINER_TAG}/test"

# ローカル出力先
MODEL_OUTPUT_DIR = "output/model"
EVAL_OUTPUT_DIR  = "/tmp/notebook-pytorch-eval"
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

print(f"Dataset bucket : {DATASET_BUCKET}")
print(f"Model bucket   : {MODEL_BUCKET}")
print(f"Train data S3  : s3://{DATASET_BUCKET}/{S3_TRAIN_PREFIX}/")
print(f"Test data S3   : s3://{DATASET_BUCKET}/{S3_TEST_PREFIX}/")

---
## 2. データの読み込みと確認

S3 から学習データとテストデータを直接メモリに読み込みます。

In [ ]:
s3 = boto3.client("s3", region_name=REGION)

def load_csv_from_s3(bucket, prefix):
    """S3 プレフィックス配下の CSV ファイルを読み込んで結合する"""
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix + "/")
    keys = [obj["Key"] for obj in resp.get("Contents", []) if obj["Key"].endswith(".csv")]
    if not keys:
        raise FileNotFoundError(f"No CSV files found in s3://{bucket}/{prefix}/. Run 01-upload-dataset.sh first.")
    print(f"Found {len(keys)} CSV file(s) in s3://{bucket}/{prefix}/")
    dfs = []
    for key in keys:
        obj = s3.get_object(Bucket=bucket, Key=key)
        dfs.append(pd.read_csv(io.BytesIO(obj["Body"].read())))
    return pd.concat(dfs, ignore_index=True)

df_train = load_csv_from_s3(DATASET_BUCKET, S3_TRAIN_PREFIX)
df_test  = load_csv_from_s3(DATASET_BUCKET, S3_TEST_PREFIX)

print(f"\nTrain shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")
print(f"Columns: {list(df_train.columns)}")
print(f"\nTarget distribution (train):")
for label, count in df_train.iloc[:, -1].value_counts().sort_index().items():
    print(f"  Class {label}: {count} samples")
df_train.head()

データを特徴量とターゲットに分割し、PyTorch の Tensor に変換します。

In [ ]:
# 学習データ
X_train = df_train.iloc[:, :-1].values.astype(np.float32)
y_train = df_train.iloc[:, -1].values.astype(np.int64)

# テストデータ
X_test_np = df_test.iloc[:, :-1].values.astype(np.float32)
y_test = df_test.iloc[:, -1].values.astype(np.int64)
X_test = torch.from_numpy(X_test_np)

input_dim = X_train.shape[1]
num_classes = len(np.unique(y_train))

print(f"Input dim:   {input_dim}")
print(f"Num classes: {num_classes}")
print(f"Train:       {len(X_train)} samples")
print(f"Test:        {len(X_test)} samples")

---
## 3. モデル定義

`train.py` と同じ `SimpleClassifier` アーキテクチャを定義します。入力次元 → 64 → 32 → クラス数 の 3 層 MLP で、各隠れ層に Dropout(0.2) を挿入して過学習を抑制します。

モデルのアーキテクチャを変更した場合は、`train.py` と `evaluate.py` も同様に更新してください。

In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.net(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = SimpleClassifier(input_dim, num_classes).to(device)
print(model)

---
## 4. ローカル学習

Adam オプティマイザと CrossEntropyLoss で学習します。`params` のハイパーパラメータを変更してこのセルを再実行することで、異なる設定での学習結果を比較できます。

- `epochs`: 学習エポック数。増やすと精度が上がりやすいが、過学習に注意
- `batch_size`: ミニバッチサイズ。GPU メモリに応じて調整してください
- `learning_rate`: Adam オプティマイザの学習率

In [ ]:
# === ハイパーパラメータ ===
params = {
    "epochs": 20,
    "batch_size": 32,
    "learning_rate": 0.001,
}

dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
loader = DataLoader(dataset, batch_size=params["batch_size"], shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=params["learning_rate"])

model.train()
for epoch in range(params["epochs"]):
    total_loss = 0.0
    correct = 0
    total = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        out = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
        correct += (out.argmax(dim=1) == y_batch).sum().item()
        total += X_batch.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total
    print(f"Epoch {epoch + 1}/{params['epochs']} - loss: {avg_loss:.4f}, accuracy: {accuracy:.4f}")

学習データ全体に対する最終メトリクスを算出し、モデルを S3 に保存します。

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

train_accuracy = (all_preds == all_labels).mean()
train_precision = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
train_recall = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
train_f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

print(f"Train accuracy:  {train_accuracy:.4f}")
print(f"Train precision: {train_precision:.4f}")
print(f"Train recall:    {train_recall:.4f}")
print(f"Train f1:        {train_f1:.4f}")

In [ ]:
model_path = os.path.join(MODEL_OUTPUT_DIR, "model.pth")
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "input_dim": input_dim,
        "num_classes": num_classes,
    },
    model_path,
)
print(f"Model saved to {model_path}")

S3_MODEL_KEY = "notebook/pytorch/model.pth"
s3.upload_file(model_path, MODEL_BUCKET, S3_MODEL_KEY)
print(f"Model uploaded to s3://{MODEL_BUCKET}/{S3_MODEL_KEY}")

---
## 5. ローカル評価

`torch.no_grad()` で勾配計算を無効化してテストデータに対する推論を行い、評価メトリクスを算出します。

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test.to(device)).argmax(dim=1).cpu().numpy()

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, average="weighted", zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, average="weighted", zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, average="weighted", zero_division=0)),
}
print("Evaluation Metrics:")
print(json.dumps(metrics, indent=2))

In [ ]:
print(classification_report(y_test, y_pred))

### Confusion Matrix (混同行列)

Confusion Matrix は、分類モデルの予測結果を「実際のクラス × 予測クラス」の行列で可視化したものです。

- 行 (縦軸) が実際のクラス、列 (横軸) が予測クラスを表します
- 対角線上のセル (左上→右下) が正しく分類された件数です
- 対角線以外のセルは誤分類を示します

In [ ]:
cm = confusion_matrix(y_test, y_pred)
labels = sorted(np.unique(y_test))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
eval_path = os.path.join(EVAL_OUTPUT_DIR, "evaluation.json")
with open(eval_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Evaluation report saved to {eval_path}")

---
## 6. MLflow 記録

MLflow App に学習・評価のメトリクスをまとめて記録します。CloudFormation スタックで MLflow App がデプロイされていない場合は自動的にスキップされます。

### 記録される情報

| 種類 | 内容 |
|------|------|
| パラメータ | `epochs`, `batch_size`, `learning_rate` |
| メトリクス | `train_accuracy`, `train_precision`, `train_recall`, `train_f1`, `train_loss`, テスト評価メトリクス |
| アーティファクト | 学習済みモデル (`pytorch-model`)、評価レポート (`evaluation.json`) |

In [ ]:
mlflow_app_arn = ""
mlflow_app_url = ""
try:
    cfn_client = boto3.client("cloudformation", region_name=REGION)
    resp = cfn_client.describe_stacks(StackName=f"{PROJECT_NAME}-stack")
    for output in resp["Stacks"][0]["Outputs"]:
        if output["OutputKey"] == "MlflowAppArn":
            mlflow_app_arn = output["OutputValue"]
            mlflow_app_url = f"https://{mlflow_app_arn.split('/')[-1]}.mlflow.sagemaker.{REGION}.amazonaws.com"
            break
    if mlflow_app_arn:
        print(f"MLflow ARN: {mlflow_app_arn}")
        print(f"MLflow URL: {mlflow_app_url}")
except Exception as e:
    print(f"MLflow App not found ({e}). Skipping MLflow logging.")

In [ ]:
if mlflow_app_arn:
    import mlflow
    from mlflow.models import infer_signature

    logging.getLogger("mlflow.pytorch").setLevel(logging.CRITICAL)

    mlflow.set_tracking_uri(mlflow_app_arn)
    mlflow.set_experiment("pytorch-notebook")

    with mlflow.start_run():
        mlflow.log_params(params)
        mlflow.log_metric("train_accuracy", train_accuracy)
        mlflow.log_metric("train_precision", train_precision)
        mlflow.log_metric("train_recall", train_recall)
        mlflow.log_metric("train_f1", train_f1)
        mlflow.log_metric("train_loss", avg_loss)
        mlflow.log_metrics(metrics)
        mlflow.set_tag("dataset", "test")

        sample_input = torch.from_numpy(X_train[:5]).to(device)
        with torch.no_grad():
            sample_output = model(sample_input).cpu().numpy()
        signature = infer_signature(X_train[:5], sample_output)

        mlflow.pytorch.log_model(
            pytorch_model=model,
            artifact_path="pytorch-model",
            signature=signature,
        )
        mlflow.log_artifact(eval_path)
    print("Metrics logged to MLflow.")
else:
    print("Skipped (no MLflow App).")

---
## 7. SageMaker Training Job

`PyTorch` Estimator を使って SageMaker マネージドコンテナで学習ジョブを実行します。

### ローカル学習との違い

- AWS マネージドの PyTorch DLC (Deep Learning Container) を使用するため、Docker イメージのビルドが不要
- GPU インスタンスを使用することで、大規模データでの学習を高速化できます
- `hyperparameters` で渡した値は `train.py` 内の `argparse` で受け取ります
- 学習ログは CloudWatch Logs に自動保存されます
- 学習済みモデルは `output_path` に指定した S3 パスに `model.tar.gz` として保存されます

### インスタンスタイプについて

CPU インスタンス (`ml.c7i.xlarge`) を使用します。サンプルデータは小規模なため CPU で十分です。GPU が必要な場合は `ml.g6.xlarge` に変更してください。

In [ ]:
import sagemaker
from sagemaker.pytorch.estimator import PyTorch as PyTorchEstimator
from sagemaker.inputs import TrainingInput

sm_session = sagemaker.Session(boto_session=boto3.Session(region_name=REGION))
role = sagemaker.get_execution_role()

print(f"SageMaker role: {role}")
print(f"SageMaker SDK version: {sagemaker.__version__}")

In [ ]:
pytorch_estimator = PyTorchEstimator(
    entry_point="train.py",
    source_dir=CONTAINER_DIR,
    framework_version="2.5.1",
    py_version="py311",
    role=role,
    instance_type="ml.c7i.xlarge",
    instance_count=1,
    output_path=f"s3://{MODEL_BUCKET}/sagemaker-jobs/pytorch",
    hyperparameters={
        "epochs": 20,
        "batch-size": 32,
        "learning-rate": 0.001,
    },
    environment={
        "MLFLOW_APP_ARN": mlflow_app_arn,
        "MLFLOW_APP_URL": mlflow_app_url,
    },
    sagemaker_session=sm_session,
)

print("PyTorch Estimator defined.")

In [ ]:
pytorch_estimator.fit(
    # input_mode: "FastFile" = S3 からオンデマンドストリーミング / "File" = 全量ダウンロード後に学習開始
    inputs={"train": TrainingInput(s3_data=f"s3://{DATASET_BUCKET}/{S3_TRAIN_PREFIX}/", input_mode="FastFile")},
    wait=True,
    logs="All",
)

print(f"Training job name: {pytorch_estimator.latest_training_job.name}")
print(f"Model artifacts:   {pytorch_estimator.model_data}")

---
## 8. SageMaker Processing Job

`PyTorchProcessor` を使って SageMaker マネージドコンテナで評価ジョブを実行します。

### 処理の流れ

1. 最新の Training Job から `model.tar.gz` の S3 URI を動的に取得
2. Processing Job がモデルとテストデータをコンテナにダウンロード
3. `evaluate.py` がモデルを読み込み、テストデータで推論・評価
4. 評価結果 (`evaluation.json`) を S3 に出力

### 入出力のマッピング

| 入出力 | S3 パス | コンテナ内パス |
|--------|---------|---------------|
| モデル (入力) | Training Job の `model.tar.gz` | `/opt/ml/processing/model` |
| テストデータ (入力) | `s3://{bucket}/container-pytorch-dlc/test/` | `/opt/ml/processing/test` |
| 評価結果 (出力) | `s3://{bucket}/notebook-pytorch/` | `/opt/ml/processing/evaluation` |

In [ ]:
from sagemaker.pytorch.processing import PyTorchProcessor
from sagemaker.s3 import S3Downloader

# Section 7 の Estimator から model.tar.gz の S3 URI を取得
MODEL_S3_URI = pytorch_estimator.model_data

EVAL_S3_URI = f"s3://{PROJECT_NAME}-eval-{boto3.client('sts').get_caller_identity()['Account']}-{REGION}/notebook-pytorch"

print(f"Model S3:       {MODEL_S3_URI}")
print(f"Eval output S3: {EVAL_S3_URI}")

In [ ]:
eval_processor = PyTorchProcessor(
    framework_version="2.5.1",
    py_version="py311",
    role=role,
    instance_count=1,
    instance_type="ml.c7i.xlarge",
    env={"MLFLOW_APP_ARN": mlflow_app_arn},
    sagemaker_session=sm_session,
)

eval_processor.run(
    code="evaluate.py",
    source_dir=CONTAINER_DIR,
    inputs=[
        sagemaker.processing.ProcessingInput(
            source=MODEL_S3_URI,
            destination="/opt/ml/processing/model",
        ),
        sagemaker.processing.ProcessingInput(
            source=f"s3://{DATASET_BUCKET}/{S3_TEST_PREFIX}",
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        sagemaker.processing.ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation",
            destination=EVAL_S3_URI,
        ),
    ],
    wait=True,
    logs=True,
)

print(f"Processing job completed. Results saved to {EVAL_S3_URI}")

In [ ]:
eval_files = S3Downloader.list(EVAL_S3_URI)
for s3_uri in eval_files:
    if s3_uri.endswith("evaluation.json"):
        content = S3Downloader.read_file(s3_uri)
        result = json.loads(content)
        print("SageMaker Processing Job - Evaluation Metrics:")
        print(json.dumps(result, indent=2))
        break
else:
    print("evaluation.json not found in output.")

---
## 9. SageMaker Pipeline

Train → Register → Evaluate の 3 ステップを SageMaker Pipeline として一括実行します。

### Pipeline とは

SageMaker Pipeline は、ML ワークフローを再現可能な形で管理するためのサービスです。各ステップ (Training Job、Processing Job、Model Registry 登録) を DAG (有向非巡回グラフ) として定義し、ステップ間のデータ依存を自動解決します。

### 処理の流れ

1. **Train Step**: PyTorch DLC で `train.py` を実行し、モデルを学習。出力は `model.tar.gz` として S3 に保存
2. **Register Step**: Train Step の出力モデルを SageMaker Model Registry に登録
3. **Evaluate Step**: Train Step の出力モデルをテストデータで評価し、`evaluation.json` を S3 に出力

### PipelineSession について

`PipelineSession` は Pipeline 定義時に使用する特殊なセッションです。通常の `Session` と異なり、Estimator や Processor の `fit()` / `run()` を呼んでも実際のジョブは起動せず、Pipeline のステップ定義として記録されます。実際のジョブ実行は `pipeline.start()` 時に行われます。

In [ ]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import TrainingStep, ProcessingStep
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.processing import FrameworkProcessor
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.pipeline_context import PipelineSession

account_id = boto3.client("sts").get_caller_identity()["Account"]
pipeline_session = PipelineSession()

ROLE_ARN = outputs.get("SageMakerRoleArn", role)

dataset_bucket = DATASET_BUCKET
train_data_uri = f"s3://{dataset_bucket}/{S3_TRAIN_PREFIX}"
test_data_uri  = f"s3://{dataset_bucket}/{S3_TEST_PREFIX}"
model_output_uri = f"s3://{PROJECT_NAME}-model-{account_id}-{REGION}/output"
eval_output_uri  = f"s3://{PROJECT_NAME}-eval-{account_id}-{REGION}/output"

MODEL_GROUP_NAME = f"{PROJECT_NAME}-pytorch"

print(f"Role ARN:     {ROLE_ARN}")
print(f"Model group:  {MODEL_GROUP_NAME}")

### Train Step

`PyTorch` Estimator で Training Job のステップを定義します。`metric_definitions` で指定した正規表現パターンに一致するログ出力が CloudWatch Metrics に自動送信されます。`sagemaker_session=pipeline_session` を指定することで、実際のジョブ実行は Pipeline 起動時まで遅延されます。

In [ ]:
metric_definitions = [
    {"Name": "train:accuracy", "Regex": r"Training accuracy: ([0-9.]+)"},
    {"Name": "train:precision", "Regex": r"Training precision: ([0-9.]+)"},
    {"Name": "train:recall", "Regex": r"Training recall: ([0-9.]+)"},
    {"Name": "train:f1", "Regex": r"Training f1: ([0-9.]+)"},
]

from sagemaker.pytorch.estimator import PyTorch

pipeline_estimator = PyTorch(
    entry_point="train.py",
    source_dir=CONTAINER_DIR,
    framework_version="2.5.1",
    py_version="py311",
    role=ROLE_ARN,
    instance_count=1,
    instance_type="ml.c7i.xlarge",
    output_path=model_output_uri,
    hyperparameters={
        "epochs": 20,
        "batch-size": 32,
        "learning-rate": 0.001,
    },
    environment={
        "MLFLOW_APP_ARN": mlflow_app_arn,
        "MODEL_GROUP_NAME": MODEL_GROUP_NAME,
    },
    metric_definitions=metric_definitions,
    sagemaker_session=pipeline_session,
)

train_step = TrainingStep(
    name="Train",
    estimator=pipeline_estimator,
    # input_mode: \"FastFile\" = S3 からオンデマンドストリーミング / \"File\" = 全量ダウンロード後に学習開始
    inputs={"train": TrainingInput(s3_data=train_data_uri, input_mode="FastFile")},
)

print("Train step created.")

### Register Step

Train Step で出力されたモデルアーティファクト (`model.tar.gz`) を SageMaker Model Registry に登録します。`train_step.properties.ModelArtifacts.S3ModelArtifacts` でステップ間のデータ依存を定義しており、Pipeline 実行時に動的に S3 パスが解決されます。

In [ ]:
register_step = RegisterModel(
    name="RegisterModel",
    estimator=pipeline_estimator,
    model_data=train_step.properties.ModelArtifacts.S3ModelArtifacts,
    content_types=["application/json"],
    response_types=["application/json"],
    inference_instances=["ml.c7i.xlarge"],
    transform_instances=["ml.c7i.xlarge"],
    model_package_group_name=MODEL_GROUP_NAME,
)

print("Register step created.")

### Evaluate Step

Train Step で学習したモデルをテストデータで評価します。`PyTorchProcessor` で Processing Job を定義し、`evaluate.py` を実行します。`add_depends_on([register_step])` で Register Step の完了後に実行されるよう順序を制御しています。

In [ ]:
pipeline_eval_processor = PyTorchProcessor(
    framework_version="2.5.1",
    py_version="py311",
    role=ROLE_ARN,
    instance_count=1,
    instance_type="ml.c7i.xlarge",
    env={"MLFLOW_APP_ARN": mlflow_app_arn},
    sagemaker_session=pipeline_session,
)

eval_args = pipeline_eval_processor.run(
    inputs=[
        sagemaker.processing.ProcessingInput(
            source=train_step.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        sagemaker.processing.ProcessingInput(
            source=test_data_uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        sagemaker.processing.ProcessingOutput(
            output_name="evaluation",
            source="/opt/ml/processing/evaluation",
            destination=eval_output_uri,
        ),
    ],
    code="evaluate.py",
    source_dir=CONTAINER_DIR,
)

eval_step = ProcessingStep(name="Evaluate", step_args=eval_args)
eval_step.add_depends_on([register_step])

print("Evaluate step created.")

### Pipeline の作成・実行

`upsert()` で Pipeline を SageMaker に登録 (既存の場合は更新) し、`start()` で実行します。Pipeline 定義は JSON としてシリアライズされ、SageMaker サービス側で管理されます。

In [ ]:
pipeline_name = f"{PROJECT_NAME}-pytorch-pipeline"

pipeline = Pipeline(
    name=pipeline_name,
    steps=[train_step, register_step, eval_step],
    sagemaker_session=pipeline_session,
)

pipeline.upsert(role_arn=ROLE_ARN)
print(f"Pipeline '{pipeline_name}' created/updated.")

In [ ]:
execution = pipeline.start()
print(f"Pipeline execution started: {execution.arn}")
print("Waiting for completion...")

execution.wait()
print(f"Pipeline execution status: {execution.describe()['PipelineExecutionStatus']}")

### 評価結果の確認

Evaluate Step が S3 に出力した `evaluation.json` をダウンロードして表示します。

In [ ]:
eval_files = S3Downloader.list(eval_output_uri)
print(f"Evaluation output files: {eval_files}")

for s3_uri in eval_files:
    if s3_uri.endswith("evaluation.json"):
        content = S3Downloader.read_file(s3_uri)
        result = json.loads(content)
        print("\nPipeline Evaluation Metrics:")
        print(json.dumps(result, indent=2))
        break
else:
    print("evaluation.json not found in output.")

---
## Pipeline 実行 (CLI、オプション)

上記の学習・評価を SageMaker Pipeline として一括実行することもできます。

```bash
# ターミナルから実行
./pipelines/scripts/run-pipeline.sh -c container-pytorch-dlc
```

In [ ]:
# Pipeline 実行 (CLI 経由)
# !cd .. && ./pipelines/scripts/run-pipeline.sh -c container-pytorch-dlc